In [1]:
async def fake_graph_events():
    yield {"event": "on_chain_start", "name": "classifier", "data": {}}
    yield {"event": "on_chat_model_stream", "name": "model", "data": {"chunk": "A"}}
    yield {"event": "on_chain_end", "name": "classifier", "data": {"output": {"intent": "chat"}}}
    yield {"event": "on_chain_start", "name": "chat", "data": {}}
    yield {"event": "on_chain_end", "name": "chat", "data": {"output": {"chat_reply": "你好"}}}

In [2]:
OUR_NODES = {
    "classifier", "chat", "patch", "planning_research", "web_research",
    "planner", "plan_validator", "plan_review", "plan_executor",
    "architecture", "material_plan", "skeleton", "merge",
    "final_validate", "callback",
}

async def collect_node_boundaries(event_stream, allowed_nodes: set[str]) -> list[dict]:
    boundaries = []
    async for event in event_stream:
        kind = event.get("event")
        node_name = event.get("name", "")
        if node_name not in allowed_nodes:
            continue
        if kind not in {"on_chain_start", "on_chain_end"}:
            continue
        boundaries.append({"kind": kind, "node": node_name, "data": event.get("data", {})})
    return boundaries

boundaries = await collect_node_boundaries(fake_graph_events(), OUR_NODES)
boundaries

[{'kind': 'on_chain_start', 'node': 'classifier', 'data': {}},
 {'kind': 'on_chain_end',
  'node': 'classifier',
  'data': {'output': {'intent': 'chat'}}},
 {'kind': 'on_chain_start', 'node': 'chat', 'data': {}},
 {'kind': 'on_chain_end',
  'node': 'chat',
  'data': {'output': {'chat_reply': '你好'}}}]

In [3]:
compact_path = [(item["kind"], item["node"]) for item in boundaries]

assert compact_path == [
    ("on_chain_start", "classifier"),
    ("on_chain_end", "classifier"),
    ("on_chain_start", "chat"),
    ("on_chain_end", "chat"),
]
print(compact_path)

[('on_chain_start', 'classifier'), ('on_chain_end', 'classifier'), ('on_chain_start', 'chat'), ('on_chain_end', 'chat')]


In [4]:
def to_agent_step(boundary: dict, request_id: str) -> dict:
    is_start = boundary["kind"] == "on_chain_start"
    return {
        "type": "agent_step",
        "request_id": request_id,
        "node": boundary["node"],
        "status": "running" if is_start else "done",
    }

agent_steps = [to_agent_step(item, "req-1") for item in boundaries]
agent_steps

[{'type': 'agent_step',
  'request_id': 'req-1',
  'node': 'classifier',
  'status': 'running'},
 {'type': 'agent_step',
  'request_id': 'req-1',
  'node': 'classifier',
  'status': 'done'},
 {'type': 'agent_step',
  'request_id': 'req-1',
  'node': 'chat',
  'status': 'running'},
 {'type': 'agent_step',
  'request_id': 'req-1',
  'node': 'chat',
  'status': 'done'}]